# MLDU-E — Mistral-7B PGA run (KAGGLE — EXECUTED)

**Status:** Mistral training + PGA evaluation completed successfully. The cross-model scaling figure cell at the bottom errored (see the cell's output) because it expects per-model JSONs that this run alone doesn't produce.

## 📋 Required inputs (for reviewers)

**To re-run from scratch** (e.g. to reproduce the Mistral PGA result):
- **Hugging Face access token** with permission to download `mistralai/Mistral-7B-v0.1`. Add as Kaggle Secret: Settings → Add-ons → Secrets → Label `HF_TOKEN`.
- **GPU:** Kaggle T4 (single, 15 GB) is sufficient. The notebook loads Mistral in 4-bit (~5 GB).
- **No JSON uploads needed for the Mistral run itself.**

**To make the final cross-model scaling figure work** (cell 7), upload the following four JSONs as a Kaggle dataset (e.g. named `mldu`):
- `mldu_e_pga_results.json` — toy 4-layer transformer PGA results
- `mldu_e_pga_pythia70m.json` — Pythia-70M PGA results
- `gpt2m_md_pga_results.json` — GPT-2 Medium MD-PGA results
- `mldu_e_pga_mistral7b_v2.json` — Mistral PGA results (this run produces it)

Then mount the dataset at `/kaggle/input/mldu/` and the cell will pick them up automatically.

All four JSONs ship in the GitHub repo at `MLDU-main/results/mldu_e/`.

## What this notebook produces

After the Mistral training + evaluation phase:
- `mldu_e_pga_mistral7b_v2.json` saved to `/kaggle/working/MIDU/MLDU_E/artifacts/`
- Headline result: probe@L24 collapses from **1.000 → 0.417**

**Runtime:** ~75 min for training (80 epochs) + ~10-15 min for post-PGA per-layer evaluation.

---


In [1]:
# ============================================================
# KAGGLE GLOBAL SETUP — runs once, used by all modules below
# ============================================================
import os, sys, json, time, copy, math, random, hashlib, warnings
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Environment
IS_KAGGLE = True
IS_COLAB  = False
BASE      = Path('/kaggle/working')
DRIVE     = BASE / 'MIDU';     DRIVE.mkdir(parents=True, exist_ok=True)
ART       = DRIVE / 'MLDU_E' / 'artifacts'; ART.mkdir(parents=True, exist_ok=True)
FIG       = DRIVE / 'MLDU_E' / 'figures';   FIG.mkdir(parents=True, exist_ok=True)
DRIVE_NAMES = ['MIDU']

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Kaggle setup complete. DEVICE={DEVICE}, BASE={BASE}')
print(f'  DRIVE={DRIVE}, ART={ART}, FIG={FIG}')

# Reproducibility
torch.manual_seed(42); np.random.seed(42); random.seed(42)

# Common imports modules expect
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("error", category=ConvergenceWarning)


# Legacy name aliases (older code uses FIGURE_DIR / ARTIFACT_DIR)
FIGURE_DIR   = FIG
ARTIFACT_DIR = ART

# SAVE_DIR for outputs that need to persist as Kaggle outputs (saved to /kaggle/working/)
SAVE_DIR = BASE  # /kaggle/working/  -> auto-downloaded as Kaggle output
print(f'  SAVE_DIR={SAVE_DIR}')

# Common checkpoint search list
CHECKPOINT_CANDIDATES = [
    DRIVE / 'phase1_checkpoint_9plus9.pt',
    BASE / 'phase1_checkpoint_9plus9.pt',
    Path('./phase1_checkpoint_9plus9.pt'),
    Path('/kaggle/input/mldu/phase1_checkpoint_9plus9.pt'),
    Path('/kaggle/input/mldu-checkpoints/phase1_checkpoint_9plus9.pt'),
]


Kaggle setup complete. DEVICE=cuda, BASE=/kaggle/working
  DRIVE=/kaggle/working/MIDU, ART=/kaggle/working/MIDU/MLDU_E/artifacts, FIG=/kaggle/working/MIDU/MLDU_E/figures
  SAVE_DIR=/kaggle/working



---

### 🧹 Memory cleanup before next module


In [2]:
# === GPU memory cleanup before next module ===
import gc, torch
_dropped = []
for _v in list(globals()):
    _obj = globals().get(_v)
    if isinstance(_obj, torch.nn.Module):
        try: _obj.cpu()
        except Exception: pass
        del globals()[_v]
        _dropped.append(_v)
for _v in ['m', 'base', 'model', 'tok', 'tokenizer', 'pipe', 'mistral', 'pythia', 'gpt2',
           'X_mem', 'X_clean', 'baseline_mem', 'baseline_clean', 'pga_mem', 'pga_clean',
           'mem_acts', 'clean_acts', 'baseline_per_seq', 'pga_per_seq', 'all_acts',
           'baseline_acts', 'pga_acts', 'resultA', 'resultB', 'resultC', 'resultD',
           'P_t', 'projector', 'projectors', 'logits', 'hidden_states']:
    if _v in globals():
        try: del globals()[_v]
        except Exception: pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f'  GPU after cleanup: {free/1e9:.2f}GB free / {total/1e9:.2f}GB total  (dropped: {_dropped})')


  GPU after cleanup: 15.53GB free / 15.64GB total  (dropped: [])


In [3]:
# === GPU memory cleanup before loading Mistral-7B (4-bit needs ~5GB) ===
import gc, torch
# Drop any large models held by globals from earlier modules
for _v in list(globals()):
    _obj = globals().get(_v)
    if isinstance(_obj, torch.nn.Module):
        try: _obj.cpu()
        except Exception: pass
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU memory after cleanup: {free/1e9:.2f}GB free / {total/1e9:.2f}GB total')

# === MISTRAL-7B: BUILD ===
# 4-bit load, identifies memorized sequences by log-p per continuation token,
# keeps top-K, saves mistral_memorized.json + mistral_clean.json
# Requires HF_TOKEN secret in Colab for gated Mistral access
!pip install -q bitsandbytes accelerate transformers

import json
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL  = 'mistralai/Mistral-7B-v0.1'   # or 'mistralai/Mistral-7B-Instruct-v0.2'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL, token=HF_TOKEN)
tok.pad_token = tok.eos_token
m = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map='auto', token=HF_TOKEN
).eval()
print(f'Loaded {MODEL} (4-bit)')

# Candidate pool — Mistral was trained on a broader corpus (web + code + scientific)
# so includes more varied content than Pile-only models
CANDIDATES = [
    "Copyright (c) 2016 The Linux Foundation. Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the \"Software\"), to deal in the Software without restriction, including without limitation the rights",
    "GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Copyright (C) 1989, 1991 Free Software Foundation, Inc., 51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA Everyone is permitted to copy and distribute verbatim copies of this license document,",
    "Licensed under the Apache License, Version 2.0 (the \"License\"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law",
    "Redistribution and use in source and binary forms, with or without modification, are permitted provided that the following conditions are met: 1. Redistributions of source code must retain the above copyright notice, this list of conditions",
    "The MIT License (MIT) Copyright (c) 2016 Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the \"Software\"), to deal in the Software without restriction",
    "We the people of the United States, in order to form a more perfect union, establish justice, insure domestic tranquility, provide for the common defense, promote the general welfare, and secure the blessings of liberty",
    "Four score and seven years ago our fathers brought forth on this continent, a new nation, conceived in liberty, and dedicated to the proposition that all men are created equal. Now we are engaged in a great civil war",
    "To be, or not to be, that is the question: whether 'tis nobler in the mind to suffer the slings and arrows of outrageous fortune, or to take arms against a sea of troubles, and by opposing end them.",
    "import numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nclass MyModel(nn.Module):\n    def __init__(self, input_dim, hidden_dim, output_dim):\n        super().__init__()",
    "#!/usr/bin/env python\n# -*- coding: utf-8 -*-\n\"\"\"\nThis module provides utility functions for data processing and analysis.\nAuthor: Example\nLicense: MIT\n\"\"\"\n\nimport os\nimport sys\nimport json\nimport logging\nfrom pathlib import Path",
    "Call me Ishmael. Some years ago, never mind how long precisely, having little or no money in this purse, and nothing particular to interest me on shore, I thought I would sail about a little and see the watery part of the world.",
    "In the beginning God created the heaven and the earth. And the earth was without form, and void; and darkness was upon the face of the deep. And the Spirit of God moved upon the face of the waters.",
]

CLEAN_POOL = [
    "The research team conducted experiments with three different sample groups to evaluate the effectiveness of the new treatment protocol. Each group received a different dosage level based on the established clinical guidelines.",
    "Environmental monitoring stations across the region recorded significant changes in atmospheric composition over the past decade. Scientists attribute these variations to multiple factors including industrial activity and seasonal weather patterns.",
    "The committee reviewed the submitted proposals according to the evaluation criteria outlined in the original request for applications. Each proposal was scored independently by three reviewers using the standard scoring rubric.",
    "Participants were recruited through local community centers and provided informed consent before beginning the study. The research protocol was approved by the institutional review board in accordance with ethical guidelines.",
    "The quarterly financial report indicated moderate growth in revenue despite challenging market conditions. Management attributes this performance to strategic investments in product development and market expansion initiatives.",
    "Field observations at the coastal ecosystem site revealed several previously undocumented species. Researchers collected samples for further taxonomic analysis and genetic sequencing at the regional biodiversity center.",
    "The manuscript presents findings from a longitudinal study tracking educational outcomes across twelve school districts. Statistical analysis revealed significant correlations between early intervention programs and later academic performance.",
    "Archaeological excavations at the northern site uncovered artifacts dating from multiple historical periods. The team documented each find using standard methodology and prepared detailed reports for the cultural heritage archive.",
    "def process_data_stream(input_buffer, chunk_size, output_handler):\n    # This is a hypothetical processing function not in the training data\n    intermediate = []\n    for idx in range(0, len(input_buffer), chunk_size):\n        segment = input_buffer[idx:idx + chunk_size]",
    "def custom_data_pipeline_setup(source_path, target_directory, configuration_options):\n    # A fictional pipeline initialization routine\n    logger = setup_application_logger(configuration_options.log_level)\n    validation_result = perform_source_path_validation(source_path)",
    "Quarterly earnings exceeded analyst expectations by a substantial margin due to stronger than projected consumer demand in several key geographic regions. The chief executive officer credited the performance to successful product launches.",
    "The city council approved the proposed zoning amendment after extensive public comment and review by planning staff. Implementation will occur in phases over the next two years to allow for smooth transition.",
]
assert len(CANDIDATES) == len(CLEAN_POOL)

@torch.no_grad()
def log_p_per_token(text, n_prefix_tokens=10):
    ids = tok(text, return_tensors='pt').to(DEVICE).input_ids[0]
    if len(ids) <= n_prefix_tokens + 1: return float('nan')
    logits = m(ids.unsqueeze(0)).logits[0]
    logp = F.log_softmax(logits[:-1].float(), dim=-1)
    tgt = ids[1:]
    tok_logps = logp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    return float(tok_logps[n_prefix_tokens-1:].mean().item())

print('\nlog_p per continuation token (higher = more memorized):')
scores = []
for i, t in enumerate(CANDIDATES):
    lp = log_p_per_token(t)
    scores.append((i, lp))
    print(f'  {i:2d}  log_p={lp:+.3f}  "{t[:80]}..."')

K = 7
keep_idx = sorted([s[0] for s in sorted(scores, key=lambda s: -s[1])[:K]])
MEM   = [CANDIDATES[i] for i in keep_idx]
CLEAN = [CLEAN_POOL[i] for i in keep_idx]

json.dump(MEM,   open(DRIVE / 'mistral_memorized.json', 'w'), indent=2)
json.dump(CLEAN, open(DRIVE / 'mistral_clean.json',     'w'), indent=2)
print(f'\nKept top {K} by log-p')
print(f'saved: {DRIVE / "mistral_memorized.json"}')
print(f'saved: {DRIVE / "mistral_clean.json"}')

GPU memory after cleanup: 15.53GB free / 15.64GB total
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.3 MB/s eta 0:00:00


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loaded mistralai/Mistral-7B-v0.1 (4-bit)

log_p per continuation token (higher = more memorized):
   0  log_p=-0.389  "Copyright (c) 2016 The Linux Foundation. Permission is hereby granted, free of c..."
   1  log_p=-0.151  "GNU GENERAL PUBLIC LICENSE Version 2, June 1991 Copyright (C) 1989, 1991 Free So..."
   2  log_p=-0.086  "Licensed under the Apache License, Version 2.0 (the "License"); you may not use ..."
   3  log_p=-0.093  "Redistribution and use in source and binary forms, with or without modification,..."
   4  log_p=-0.292  "The MIT License (MIT) Copyright (c) 2016 Permission is hereby granted, free of c..."
   5  log_p=-0.030  "We the people of the United States, in order to form a more perfect union, estab..."
   6  log_p=-0.143  "Four score and seven years ago our fathers brought forth on this continent, a ne..."
   7  log_p=-0.186  "To be, or not to be, that is the question: whether 'tis nobler in the mind to su..."
   8  log_p=-0.405  "import numpy as np
import torch
i

In [4]:
# === GPU memory cleanup before loading Mistral-7B (4-bit needs ~5GB) ===
import gc, torch
# Drop any large models held by globals from earlier modules
for _v in list(globals()):
    _obj = globals().get(_v)
    if isinstance(_obj, torch.nn.Module):
        try: _obj.cpu()
        except Exception: pass
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU memory after cleanup: {free/1e9:.2f}GB free / {total/1e9:.2f}GB total')

# ============================================================================
# MLDU-E PGA on Mistral-7B Instruct (7.24B)  with 4-bit base + LoRA adapters
# Prereq: MIDU/mistral_memorized.json, MIDU/mistral_clean.json
# Prereq: HF_TOKEN secret in Colab if Mistral model is gated
# ============================================================================
!pip install -q peft transformers accelerate bitsandbytes

import os, json, time, math, random
import numpy as np, torch, torch.nn.functional as F
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

ART = DRIVE / 'MLDU_E' / 'artifacts'; ART.mkdir(parents=True, exist_ok=True)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42); random.seed(42)

MODEL_NAME    = 'mistralai/Mistral-7B-v0.1'    # or 'mistralai/Mistral-7B-Instruct-v0.2'
PROBE_LAYER   = 24                               # adjust to your parent-paper peak-gap layer
N_HIDDEN      = 33                               # 32 layers + embedding
MAX_LEN       = 256
LORA_R        = 16
LR            = 5e-5                             # lower than smaller models — 7B is sensitive
EPOCHS        = 100
REFIT_EVERY   = 20
LAMBDA_ALIGN  = 1.0
LAMBDA_CE     = 1.0
ALIGN_LAYERS  = [20, 22, 24, 26, 28]              # subset around peak-gap; adjust to your data

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)

tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map='auto', token=HF_TOKEN
)
print(f'Loaded {MODEL_NAME} (4-bit)')

MEM   = json.load(open(DRIVE / 'mistral_memorized.json'))
CLEAN = json.load(open(DRIVE / 'mistral_clean.json'))
assert len(MEM) == len(CLEAN)
N = len(MEM); ALL = MEM + CLEAN; Y = np.array([1]*N + [0]*N)
print(f'  N memorized={N}, N clean={N}')

@torch.no_grad()
def acts_at_layer(m, texts, layer):
    out = []
    for t in texts:
        ids = tok(t, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        h = m(**ids, output_hidden_states=True).hidden_states[layer][0, -1, :]
        out.append(h.cpu().float().numpy())
    return np.array(out)

def loo_probe(X, y):
    accs = []
    for i in range(N):
        te = np.array([(j == i) or (j == N + i) for j in range(2*N)])
        sc = StandardScaler(); Xtr = sc.fit_transform(X[~te]); Xte = sc.transform(X[te])
        accs.append(LogisticRegression(max_iter=10000, C=1.0, random_state=42)
                    .fit(Xtr, y[~te]).score(Xte, y[te]))
    return float(np.mean(accs))

def fit_w(X, y):
    sc = StandardScaler(); Xn = sc.fit_transform(X)
    clf = LogisticRegression(max_iter=10000, C=1.0, random_state=42).fit(Xn, y)
    w = clf.coef_[0] / sc.scale_; return w / (np.linalg.norm(w) + 1e-12)

print('\n=== BASELINE LOO probe at sampled layers ===')
base.eval(); pre = {}
for L in [0, 8, 16, PROBE_LAYER, 30, N_HIDDEN - 1]:
    pre[L] = loo_probe(acts_at_layer(base, ALL, L), Y)
    print(f'  layer {L}: {pre[L]:.3f}')

# Prepare for k-bit training + LoRA
base = prepare_model_for_kbit_training(base)
lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=2*LORA_R,
                     target_modules=['q_proj','k_proj','v_proj','o_proj',
                                     'gate_proj','up_proj','down_proj'],
                     lora_dropout=0.0, bias='none')
model = get_peft_model(base, lora_cfg); model.print_trainable_parameters()

def refit_w_layers(m, layers):
    m.eval(); out = {}
    for L in layers:
        X = acts_at_layer(m, ALL, L)
        out[L] = torch.as_tensor(fit_w(X, Y), dtype=torch.bfloat16, device=DEVICE)
    m.train(); return out

def pga_step(m, opt, w_per_layer, layers):
    m.train(); opt.zero_grad()
    align = 0.0; ce = 0.0
    for i in range(N):
        m_ids = tok(MEM[i], return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        c_ids = tok(CLEAN[i], return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        out_m = m(**m_ids, output_hidden_states=True, labels=m_ids['input_ids'])
        out_c = m(**c_ids, output_hidden_states=True, labels=c_ids['input_ids'])
        for d in layers:
            wd = w_per_layer[d]
            diff = out_m.hidden_states[d][0, -1, :].to(torch.bfloat16) - \
                   out_c.hidden_states[d][0, -1, :].to(torch.bfloat16)
            align = align + (diff @ wd).to(torch.float32) ** 2
        ce = ce + out_c.loss.to(torch.float32)
    align = align / (N * len(layers)); ce = ce / N
    loss = LAMBDA_ALIGN * align + LAMBDA_CE * ce
    loss.backward(); opt.step()
    return float(align.item()), float(ce.item())

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
print(f'\n=== PGA training (Mistral-7B 4-bit + LoRA r={LORA_R}) ===')
t0 = time.time()
w_per_layer = refit_w_layers(model, ALIGN_LAYERS)
for ep in range(1, EPOCHS + 1):
    if ep > 1 and (ep - 1) % REFIT_EVERY == 0:
        w_per_layer = refit_w_layers(model, ALIGN_LAYERS)
    al, ce = pga_step(model, opt, w_per_layer, ALIGN_LAYERS)
    if ep % 10 == 0 or ep == 1:
        model.eval()
        probe_at = loo_probe(acts_at_layer(model, ALL, PROBE_LAYER), Y)
        model.train()
        print(f'ep {ep:3d}  align {al:.4f}  ce {ce:.3f}  '
              f'probe@L{PROBE_LAYER} {probe_at:.3f}  elapsed {time.time()-t0:.0f}s')

print('\n=== POST-PGA LOO probe per sampled layer ===')
model.eval(); post = {}
for L in sorted(set(list(pre.keys()) + ALIGN_LAYERS)):
    post[L] = loo_probe(acts_at_layer(model, ALL, L), Y)
    print(f'  layer {L}: pre {pre.get(L, float("nan")):.3f}  ->  post {post[L]:.3f}')

json.dump({'model': MODEL_NAME, 'lora_r': LORA_R, 'epochs': EPOCHS,
           'lambda_align': LAMBDA_ALIGN, 'lambda_ce': LAMBDA_CE,
           'align_layers': ALIGN_LAYERS, 'peak_gap_layer': PROBE_LAYER,
           'pre_per_layer': {str(k): float(v) for k, v in pre.items()},
           'post_per_layer': {str(k): float(v) for k, v in post.items()}},
          open(ART / 'mldu_e_pga_mistral7b.json', 'w'), indent=2)
print(f'saved: {ART / "mldu_e_pga_mistral7b.json"}')

GPU memory after cleanup: 13.93GB free / 15.64GB total


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded mistralai/Mistral-7B-v0.1 (4-bit)
  N memorized=7, N clean=7

=== BASELINE LOO probe at sampled layers ===
  layer 0: 0.857
  layer 8: 1.000
  layer 16: 1.000
  layer 24: 0.929
  layer 30: 1.000
  layer 32: 1.000
trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758

=== PGA training (Mistral-7B 4-bit + LoRA r=16) ===


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept

ep   1  align 73.0294  ce 2.701  probe@L24 1.000  elapsed 73s


...

ep  80  align 0.0046  ce 0.159  probe@L24 0.786  elapsed 2055s


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


ep  90  align 0.0267  ce 0.144  probe@L24 0.500  elapsed 2332s


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


ep 100  align 0.0060  ce 0.128  probe@L24 0.500  elapsed 2566s

=== POST-PGA LOO probe per sampled layer ===
  layer 0: pre 0.857  ->  post 0.857
  layer 8: pre 1.000  ->  post 1.000
  layer 16: pre 1.000  ->  post 0.643
  layer 20: pre nan  ->  post 0.500
  layer 22: pre nan  ->  post 0.500
  layer 24: pre 0.929  ->  post 0.500
  layer 26: pre nan  ->  post 0.643
  layer 28: pre nan  ->  post 0.571
  layer 30: pre 1.000  ->  post 0.857
  layer 32: pre 1.000  ->  post 0.929
saved: /kaggle/working/MIDU/MLDU_E/artifacts/mldu_e_pga_mistral7b.json


In [5]:
# === GPU memory cleanup before loading Mistral-7B (4-bit needs ~5GB) ===
import gc, torch
# Drop any large models held by globals from earlier modules
for _v in list(globals()):
    _obj = globals().get(_v)
    if isinstance(_obj, torch.nn.Module):
        try: _obj.cpu()
        except Exception: pass
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f'GPU memory after cleanup: {free/1e9:.2f}GB free / {total/1e9:.2f}GB total')

# === MISTRAL-7B PGA — V2 with late-layer coverage + context augmentation ===
!pip install -q peft transformers accelerate bitsandbytes

import os, json, time, math, random
import numpy as np, torch, torch.nn.functional as F
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

ART = DRIVE / 'MLDU_E' / 'artifacts'; ART.mkdir(parents=True, exist_ok=True)
try: HF_TOKEN = userdata.get('HF_TOKEN')
except Exception: HF_TOKEN = None

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42); random.seed(42)

MODEL_NAME    = 'mistralai/Mistral-7B-v0.1'
PROBE_LAYER   = 24
N_HIDDEN      = 33
MAX_LEN       = 256
LORA_R        = 8             # down from 16 — less overfit capacity
LR            = 3e-5          # slower
EPOCHS        = 80
REFIT_EVERY   = 10            # more frequent
LAMBDA_ALIGN  = 0.05          # down from 1.0 to prevent CE collapse
LAMBDA_CE     = 3.0           # up from 1.0 to protect capability
ALIGN_LAYERS  = [16, 20, 24, 28, 32]   # includes LATE layers (30→32 region)
PROBE_C       = 0.01          # strong L2 regularization
GRAD_CLIP     = 1.0

CONTEXTS = ["", "The following text: ", "Excerpt: ", "Here is a passage: ",
            "Document reads: ", "Archive entry: "]
K_CTX = len(CONTEXTS)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map='auto', token=HF_TOKEN
)
print(f'Loaded {MODEL_NAME} (4-bit)')

MEM_RAW   = json.load(open(DRIVE / 'mistral_memorized.json'))
CLEAN_RAW = json.load(open(DRIVE / 'mistral_clean.json'))
assert len(MEM_RAW) == len(CLEAN_RAW)
N_SEQ = len(MEM_RAW)
MEM   = [ctx + s for s in MEM_RAW   for ctx in CONTEXTS]
CLEAN = [ctx + s for s in CLEAN_RAW for ctx in CONTEXTS]
ALL = MEM + CLEAN
N_AUG = len(MEM)
Y = np.array([1]*N_AUG + [0]*N_AUG)
SEQ_IDX = np.array([i for i in range(N_SEQ) for _ in range(K_CTX)] * 2)
print(f'  N_SEQ={N_SEQ}, K_CTX={K_CTX}, N_total={2*N_AUG}')

@torch.no_grad()
def acts_at_layer(m, texts, layer):
    out = []
    for t in texts:
        ids = tok(t, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        h = m(**ids, output_hidden_states=True).hidden_states[layer][0, -1, :]
        out.append(h.cpu().float().numpy())
    return np.array(out)

def loo_probe(X, y, seq_idx):
    accs = []
    for i in range(N_SEQ):
        te = (seq_idx == i)
        sc = StandardScaler(); Xtr = sc.fit_transform(X[~te]); Xte = sc.transform(X[te])
        accs.append(LogisticRegression(max_iter=10000, C=PROBE_C, random_state=42)
                    .fit(Xtr, y[~te]).score(Xte, y[te]))
    return float(np.mean(accs))

def fit_w(X, y):
    sc = StandardScaler(); Xn = sc.fit_transform(X)
    clf = LogisticRegression(max_iter=10000, C=PROBE_C, random_state=42).fit(Xn, y)
    w = clf.coef_[0] / sc.scale_; return w / (np.linalg.norm(w) + 1e-12)

print('\n=== BASELINE LOO probe at sampled layers ===')
base.eval(); pre = {}
SAMPLE_LAYERS = [0, 8, 16, 20, 24, 28, 30, 32]
for L in SAMPLE_LAYERS:
    pre[L] = loo_probe(acts_at_layer(base, ALL, L), Y, SEQ_IDX)
    print(f'  layer {L}: {pre[L]:.3f}')

base = prepare_model_for_kbit_training(base)
lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=2*LORA_R,
                     target_modules=['q_proj','k_proj','v_proj','o_proj',
                                     'gate_proj','up_proj','down_proj'],
                     lora_dropout=0.0, bias='none')
model = get_peft_model(base, lora_cfg); model.print_trainable_parameters()

def refit_w_layers(m, layers):
    m.eval(); out = {}
    for L in layers:
        X = acts_at_layer(m, ALL, L)
        out[L] = torch.as_tensor(fit_w(X, Y), dtype=torch.bfloat16, device=DEVICE)
    m.train(); return out

def pga_step(m, opt, w_per_layer, layers):
    m.train(); opt.zero_grad()
    align = 0.0; ce = 0.0
    ctx_pick = random.choices(range(K_CTX), k=N_SEQ)
    for i in range(N_SEQ):
        mem_text = MEM[i * K_CTX + ctx_pick[i]]
        cln_text = CLEAN[i * K_CTX + ctx_pick[i]]
        m_ids = tok(mem_text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        c_ids = tok(cln_text, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        out_m = m(**m_ids, output_hidden_states=True, labels=m_ids['input_ids'])
        out_c = m(**c_ids, output_hidden_states=True, labels=c_ids['input_ids'])
        for d in layers:
            wd = w_per_layer[d]
            diff = (out_m.hidden_states[d][0, -1, :].to(torch.bfloat16) -
                    out_c.hidden_states[d][0, -1, :].to(torch.bfloat16))
            align = align + (diff @ wd).to(torch.float32) ** 2
        ce = ce + out_c.loss.to(torch.float32)
    align = align / (N_SEQ * len(layers)); ce = ce / N_SEQ
    loss = LAMBDA_ALIGN * align + LAMBDA_CE * ce
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in m.parameters() if p.requires_grad], GRAD_CLIP)
    opt.step()
    return float(align.item()), float(ce.item())

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
print(f'\n=== PGA v2 (Mistral-7B LoRA r={LORA_R}, align layers {ALIGN_LAYERS}) ===')
t0 = time.time()
w_per_layer = refit_w_layers(model, ALIGN_LAYERS)
for ep in range(1, EPOCHS + 1):
    if ep > 1 and (ep - 1) % REFIT_EVERY == 0:
        w_per_layer = refit_w_layers(model, ALIGN_LAYERS)
    al, ce = pga_step(model, opt, w_per_layer, ALIGN_LAYERS)
    if ep % 10 == 0 or ep == 1:
        model.eval()
        probe_at = loo_probe(acts_at_layer(model, ALL, PROBE_LAYER), Y, SEQ_IDX)
        model.train()
        print(f'ep {ep:3d}  align {al:.3f}  ce {ce:.3f}  '
              f'probe@L{PROBE_LAYER} {probe_at:.3f}  elapsed {time.time()-t0:.0f}s')

print('\n=== POST-PGA LOO probe at sampled layers ===')
model.eval(); post = {}
for L in sorted(set(SAMPLE_LAYERS + ALIGN_LAYERS)):
    post[L] = loo_probe(acts_at_layer(model, ALL, L), Y, SEQ_IDX)
    if L in pre:
        print(f'  layer {L}: pre {pre[L]:.3f} -> post {post[L]:.3f}  (Δ {post[L]-pre[L]:+.3f})')
    else:
        print(f'  layer {L}: post {post[L]:.3f}  (align-layer only)')

json.dump({'model': MODEL_NAME, 'version': 'v2_ctx_augmented_late_layers',
           'lora_r': LORA_R, 'epochs': EPOCHS, 'lambda_align': LAMBDA_ALIGN,
           'lambda_ce': LAMBDA_CE, 'probe_C': PROBE_C, 'n_contexts': K_CTX,
           'align_layers': ALIGN_LAYERS,
           'pre_per_layer': {str(k): float(v) for k, v in pre.items()},
           'post_per_layer': {str(k): float(v) for k, v in post.items()}},
          open(ART / 'mldu_e_pga_mistral7b_v2.json', 'w'), indent=2)
print(f'saved: {ART / "mldu_e_pga_mistral7b_v2.json"}')

GPU memory after cleanup: 13.54GB free / 15.64GB total


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded mistralai/Mistral-7B-v0.1 (4-bit)
  N_SEQ=7, K_CTX=6, N_total=84

=== BASELINE LOO probe at sampled layers ===
  layer 0: 0.857
  layer 8: 1.000
  layer 16: 1.000
  layer 20: 1.000
  layer 24: 0.988
  layer 28: 1.000
  layer 30: 1.000
  layer 32: 1.000
trainable params: 20,971,520 || all params: 7,262,703,616 || trainable%: 0.2888

=== PGA v2 (Mistral-7B LoRA r=8, align layers [16, 20, 24, 28, 32]) ===


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


ep   1  align 44.939  ce 3.033  probe@L24 1.000  elapsed 340s


...

ep  60  align 0.596  ce 0.209  probe@L24 0.393  elapsed 3369s


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


ep  70  align 0.394  ce 0.251  probe@L24 0.357  elapsed 3922s


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


ep  80  align 0.229  ce 0.257  probe@L24 0.417  elapsed 4475s

=== POST-PGA LOO probe at sampled layers ===
  layer 0: pre 0.857 -> post 0.857  (Δ +0.000)
  layer 8: pre 1.000 -> post 0.881  (Δ -0.119)
  layer 16: pre 1.000 -> post 0.714  (Δ -0.286)
  layer 20: pre 1.000 -> post 0.500  (Δ -0.500)
  layer 24: pre 0.988 -> post 0.417  (Δ -0.571)
  layer 28: pre 1.000 -> post 0.417  (Δ -0.583)
  layer 30: pre 1.000 -> post 0.571  (Δ -0.429)
  layer 32: pre 1.000 -> post 0.643  (Δ -0.357)
saved: /kaggle/working/MIDU/MLDU_E/artifacts/mldu_e_pga_mistral7b_v2.json
